# Consistency regularization

_Semi & Self-Supervised Learning_

**A good model should not change its mind when you nudge the input. Make that a training signal and unlabeled data starts teaching.**

Consistency regularization is the backbone of modern semi-supervised learning (learning from a few labels plus a flood of unlabeled data).

       The whole idea fits in one sentence: a model's prediction should be invariant to small perturbations of the input or of the model itself. "Invariant" means it should stay the same.

---

This notebook builds that idea up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Build the student and teacher networks

Consistency methods like Mean Teacher keep **two** copies of the same network. The **student** is trained by gradient descent; the **teacher** is a slowly-updated copy that produces the targets the student tries to match. We start them identical and freeze the teacher's gradients — the teacher is never backpropped, it only trails the student. The `Dropout` layer is deliberate: it makes each forward pass a slightly different function, which is a free per-pass perturbation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- a toy classifier; swap in any backbone (ResNet, etc.) ----
def make_model(num_classes=10):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(64, 128), nn.ReLU(),
        nn.Dropout(0.3),                 # dropout = a free per-pass perturbation
        nn.Linear(128, num_classes),
    )

student = make_model().cuda()
teacher = make_model().cuda()
teacher.load_state_dict(student.state_dict())     # start identical
for p in teacher.parameters():
    p.requires_grad_(False)                       # teacher is never backpropped

opt = torch.optim.Adam(student.parameters(), lr=1e-3)

### Step 2 — Set the schedule: EMA teacher and a ramped consistency weight

Two schedules make this stable. The **EMA update** moves the teacher a tiny fraction toward the student each step (`theta' = alpha*theta' + (1-alpha)*theta`), so the teacher is a running average of recent students — smoother and a cleaner target. The **consistency weight** starts at 0 and ramps up with a Gaussian curve over the first `RAMP_EPOCHS`, so the unlabeled term does not drown the few real labels early on while the teacher is still garbage.

In [ ]:
EMA_DECAY = 0.99      # alpha in theta' = alpha*theta' + (1-alpha)*theta
RAMP_EPOCHS = 30      # ramp the unlabeled weight up slowly (avoid early garbage)
MAX_CONS_W = 10.0     # final weight on the consistency loss

def ema_update(student, teacher, alpha):
    with torch.no_grad():
        for ps, pt in zip(student.parameters(), teacher.parameters()):
            pt.mul_(alpha).add_(ps, alpha=1 - alpha)     # theta' = a*theta' + (1-a)*theta

def consistency_weight(epoch):
    # Gaussian ramp-up: 0 at start, MAX_CONS_W after RAMP_EPOCHS
    import math
    r = min(1.0, epoch / RAMP_EPOCHS)
    return MAX_CONS_W * math.exp(-5.0 * (1.0 - r) ** 2)

### Step 3 — The supervised + consistency loss on one batch

Each training step combines two terms. The **supervised** term is plain cross-entropy on the handful of labeled examples (under a weak augmentation). The **consistency** term is the heart of the method: the teacher sees a *weak* view of an unlabeled image and produces a detached target distribution; the student sees a *strong* view and must match it. We use the MSE between the two probability vectors, weighted by the ramped consistency weight — no label is involved in this second term.

In [ ]:
def train_step(x_lab, y_lab, x_unlab, weak_aug, strong_aug, epoch):
    # ---- supervised term on the few labels (weak aug on labeled data) ----
    logit_lab = student(weak_aug(x_lab))
    loss_sup = F.cross_entropy(logit_lab, y_lab)

    # ---- consistency term on unlabeled data ----
    # teacher sees the WEAK view; student sees the STRONG view
    with torch.no_grad():
        t_logits = teacher(weak_aug(x_unlab))
        t_prob = F.softmax(t_logits, dim=1)          # target distribution (detached)
    s_logits = student(strong_aug(x_unlab))
    s_prob = F.softmax(s_logits, dim=1)

    # MSE form of the consistency loss: E|| f(x+delta) - f(x) ||^2
    loss_cons = F.mse_loss(s_prob, t_prob)
    # (KL form used by UDA / VAT, equivalent intent:)
    # loss_cons = F.kl_div(F.log_softmax(s_logits, 1), t_prob, reduction="batchmean")

    w = consistency_weight(epoch)
    loss = loss_sup + w * loss_cons

### Step 4 — Backprop the student, then nudge the teacher

With the combined loss assembled, we take one ordinary optimizer step on the **student** only. Then — outside the gradient computation — we apply the EMA update so the **teacher** creeps a little toward the freshly-updated student. The function returns the two loss components and the current weight so a training loop can log them.

In [ ]:
    opt.zero_grad()
    loss.backward()
    opt.step()
    ema_update(student, teacher, EMA_DECAY)          # teacher trails the student
    return loss_sup.item(), loss_cons.item(), w

## Visualize the data & results

_On real digit images with only a handful of labels, does using the UNLABELED data lift test accuracy over training on labels alone?_

### Step 1 — Load the digits and carve out a tiny labeled budget

To see whether unlabeled data actually helps, we use the real 8x8 digits, scale the pixels to 0–1, and split off a held-out test set. We then sweep over several **label budgets** (10, 20, … 160 labeled examples), choosing each labeled subset at random with a fixed seed so the comparison is fair.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                         # pixels scaled to 0..1
y = digits.target
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

budgets = [10, 20, 40, 80, 160]

### Step 2 — Compare labels-only vs. using the unlabeled rows

For each budget we train two models on the *same* `n` labels. The **baseline** is a logistic regression on those `n` labels only — it never sees the rest. The **semi-supervised** model keeps every training row but marks the un-budgeted ones as unlabeled (`-1`) and runs `LabelSpreading`, which propagates labels across the data graph. This is the apples-to-apples test of whether unlabeled data buys you accuracy.

In [ ]:
sup, semi = [], []
rng = np.random.RandomState(0)
for n in budgets:
    perm = rng.permutation(len(ytr))
    lab = perm[:n]                             # n labeled indices

    # baseline: supervised model on the n labels ONLY
    clf = LogisticRegression(max_iter=2000).fit(Xtr[lab], ytr[lab])
    sup.append(round(accuracy_score(yte, clf.predict(Xte)), 3))

    # semi-supervised: keep ALL train rows; mark the rest unlabeled (-1)
    yt = np.full(len(ytr), -1)
    yt[lab] = ytr[lab]
    ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2).fit(Xtr, yt)
    semi.append(round(accuracy_score(yte, ls.predict(Xte)), 3))

### Step 3 — Read off the two accuracy curves

Print the two lists side by side. Especially at the smallest budgets, the with-unlabeled row should sit clearly above the labels-only row — the unlabeled images are pulling their weight. The gap narrows as labels grow, since with enough labels the supervised model can stand on its own.

In [ ]:
print("budgets:", budgets)
print("labels only      :", sup)    # [0.431, 0.702, 0.826, 0.863, 0.907]
print("with unlabeled    :", semi)  # [0.570, 0.917, 0.928, 0.894, 0.956]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** An unlabeled image gives teacher prediction $f' = [0.6, 0.3, 0.1]$ and student prediction $f = [0.4, 0.4, 0.2]$ over 3 classes. Find the squared-distance (MSE) consistency loss.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Subtract the two distributions class by class: $0.4-0.6$, $0.4-0.3$, $0.2-0.1$. — _The consistency loss measures how far apart the two predicted probability vectors are, component by component._
- Square each difference and add them up. — _$\lVert f - f'\rVert^2$ is the sum of squared differences — 0 only when the two predictions match exactly._

**Answer:** Differences: $-0.2,\ +0.1,\ +0.1$. Squares: $0.04,\ 0.01,\ 0.01$. Loss $= 0.04 + 0.01 + 0.01 = 0.06$. It is positive, so backprop nudges the student toward the teacher's $[0.6, 0.3, 0.1]$. No label was used anywhere.

</details>

**Problem 2.** The teacher's weight is currently $\theta' = 3.0$ and the student's is $\theta = 4.0$. With EMA decay $\alpha = 0.99$, what is the teacher's weight after one update? Why is the teacher's value better than just copying the student?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply $\theta'_t = \alpha\,\theta'_{t-1} + (1-\alpha)\,\theta_t = 0.99(3.0) + 0.01(4.0)$. — _The EMA keeps 99% of the old teacher and mixes in 1% of the current student, so the teacher moves slowly._
- Reason about what averaging over many steps gives you. — _A running average of many recent student weights is like an ensemble of past models — smoother and usually more accurate._

**Answer:** $\theta'_t = 0.99(3.0) + 0.01(4.0) = 2.97 + 0.04 = 3.01$. The teacher crept just $0.01$ toward the student. Because the teacher averages over many past student states, its predictions are steadier and a little more accurate than any single noisy student step — so it is a cleaner target to be consistent with, which helps avoid confirmation bias.

</details>

**Problem 3.** Your semi-supervised model collapses: it predicts the same class for every input, and the consistency loss is near 0. What two things went wrong, and how do you fix them?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice that two identical constant outputs agree perfectly. — _The consistency loss alone has a trivial minimizer — predict a constant — so it cannot be the only signal._
- Check the balance between the labeled loss and the unlabeled (consistency) loss. — _If the unlabeled term is ramped up too fast or weighted too heavily, it overpowers the few labels that pin down the real classes._

**Answer:** (1) The consistency loss has a degenerate solution — a constant prediction has 0 disagreement — so it must be combined with the supervised loss on the few labels, plus confidence thresholding or entropy minimization. (2) The unlabeled weight was likely ramped up too fast, drowning the labeled signal. Fix it by keeping the labeled loss active and ramping the unlabeled weight up slowly with a sigmoid/Gaussian schedule, and use a slow EMA teacher so the target does not chase the collapse.

</details>